# Random Forest — Sepsis Prediction Across Hospitals

In [ ]:
# Import the shared pipeline
!wget -q -O sepsis.py https://raw.githubusercontent.com/kna20/early-sepsis-prediction/main/sepsis.py

import sepsis as s
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

print("Ready for Random Forest!")

Ready for Random Forest!


In [ ]:
# Load both hospitals
data_A = s.load_hospital("A")
data_B = s.load_hospital("B")

# Recover the same 80/20 split
dev_ids, test_ids = s.split_patients(data_A)

# Prepare the three datasets using sepsis.py
dev_A = s.prepare_dev(data_A, dev_ids)
test_A = s.prepare_dev(data_A, test_ids)
test_B = s.prepare_dev(data_B, data_B["PatientID"].unique())

print("Training rows:", len(dev_A["labels"]))
print("Hospital A test rows:", len(test_A["labels"]))
print("Hospital B test rows:", len(test_B["labels"]))

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Training rows: 631120
Hospital A test rows: 159095
Hospital B test rows: 761995


In [ ]:
# Reuse the same patient-level folds as our earlier experiments
fold_of = s.make_folds(data_A, dev_ids)

# Add fold assignments to our already-prepared development data
dev_A["fold"] = pd.Series(
    dev_A["patients"]
).map(fold_of).to_numpy()

print("Development patients:", len(dev_ids))
print("Cross-validation folds:", sorted(set(dev_A["fold"])))
print("Patients per fold:")
print(fold_of.value_counts().sort_index())

Development patients: 16268
Cross-validation folds: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
Patients per fold:
1    3254
2    3254
3    3254
4    3253
5    3253
Name: count, dtype: int64


In [ ]:
# Fold 1 = validation; other folds = training
train_mask = dev_A["fold"] != 1
val_mask = dev_A["fold"] == 1

# Drop near-empty features based on training patients only
missing_pct = dev_A["missing"].loc[train_mask, s.ALL_FEATURES].mean()
keep = [c for c in s.ALL_FEATURES if missing_pct[c] < 0.99]

# Calculate missing-value replacements using training patients only
fill = (
    dev_A["filled"].loc[train_mask, keep]
    .drop(columns=s.UNIT_COLS, errors="ignore")
    .median()
    .to_dict()
)
fill.update({c: -1 for c in s.UNIT_COLS if c in keep})

# Configuration 1: Original features
X_rf_train = dev_A["filled"].loc[train_mask, keep].fillna(fill).copy()
X_rf_val = dev_A["filled"].loc[val_mask, keep].fillna(fill).copy()

# Configuration 2: Original features + missingness
clinical = [c for c in s.CLINICAL if c in keep]
flags = [c + "_measured" for c in clinical]
since = [c + "_hours_since" for c in clinical]

X_rf_train_missing = pd.concat([
    X_rf_train,
    dev_A["flags"].loc[train_mask, flags],
    dev_A["since"].loc[train_mask, since]
], axis=1)

X_rf_val_missing = pd.concat([
    X_rf_val,
    dev_A["flags"].loc[val_mask, flags],
    dev_A["since"].loc[val_mask, since]
], axis=1)

print("Original features:", X_rf_train.shape)
print("Original + missingness:", X_rf_train_missing.shape)
print("Validation:", X_rf_val_missing.shape)
print("Missing values:", X_rf_train_missing.isna().sum().sum())

Original features: (506841, 36)
Original + missingness: (506841, 96)
Validation: (124279, 96)
Missing values: 0


## Random Forest — Model Experiments

In [ ]:
import time

rf_baseline = RandomForestClassifier(
    n_estimators=60,
    max_depth=14,
    min_samples_leaf=20,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=s.SEED
)

start = time.time()

# Train on four development folds
rf_baseline.fit(
    X_rf_train,
    dev_A["labels"][train_mask]
)

# Predict on validation fold only
prob_rf_baseline = rf_baseline.predict_proba(X_rf_val)[:, 1]

print("Training time:", round(time.time() - start, 1), "seconds")

print("Validation AUROC:", round(
    roc_auc_score(dev_A["labels"][val_mask], prob_rf_baseline), 4
))

print("Validation AUPRC:", round(
    average_precision_score(dev_A["labels"][val_mask], prob_rf_baseline), 4
))

Training time: 81.3 seconds
Validation AUROC: 0.7808
Validation AUPRC: 0.0943


In [ ]:
# Second configuration: 96 features with missingness

rf_missing = RandomForestClassifier(
    n_estimators=60,
    max_depth=14,
    min_samples_leaf=20,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=s.SEED
)

start = time.time()

rf_missing.fit(
    X_rf_train_missing,
    dev_A["labels"][train_mask]
)

prob_rf_missing = rf_missing.predict_proba(
    X_rf_val_missing
)[:, 1]

print("Training + prediction time:",
      round(time.time() - start, 1), "seconds")

print("Validation AUROC:", round(
    roc_auc_score(dev_A["labels"][val_mask], prob_rf_missing), 4
))

print("Validation AUPRC:", round(
    average_precision_score(dev_A["labels"][val_mask], prob_rf_missing), 4
))

Training + prediction time: 89.7 seconds
Validation AUROC: 0.808
Validation AUPRC: 0.0949


In [ ]:
# Experiment 3: Deeper Random Forest (96 features)

rf_deep = RandomForestClassifier(
    n_estimators=60,
    max_depth=20,
    min_samples_leaf=20,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=s.SEED
)

start = time.time()

rf_deep.fit(
    X_rf_train_missing,
    dev_A["labels"][train_mask]
)

prob_rf_deep = rf_deep.predict_proba(X_rf_val_missing)[:, 1]

print("Training time:", round(time.time() - start, 1), "seconds")
print("Validation AUROC:", round(
    roc_auc_score(dev_A["labels"][val_mask], prob_rf_deep), 4
))
print("Validation AUPRC:", round(
    average_precision_score(dev_A["labels"][val_mask], prob_rf_deep), 4
))

Training time: 94.2 seconds
Validation AUROC: 0.8082
Validation AUPRC: 0.0914


In [ ]:
# Experiment 4: Smaller leaf size (96 features)

rf_leaf10 = RandomForestClassifier(
    n_estimators=60,
    max_depth=14,
    min_samples_leaf=10,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=s.SEED
)

start = time.time()

rf_leaf10.fit(
    X_rf_train_missing,
    dev_A["labels"][train_mask]
)

prob_rf_leaf10 = rf_leaf10.predict_proba(
    X_rf_val_missing
)[:, 1]

print("Training time:", round(time.time() - start, 1), "seconds")

print("Validation AUROC:", round(
    roc_auc_score(dev_A["labels"][val_mask], prob_rf_leaf10), 4
))

print("Validation AUPRC:", round(
    average_precision_score(dev_A["labels"][val_mask], prob_rf_leaf10), 4
))

Training time: 82.5 seconds
Validation AUROC: 0.8017
Validation AUPRC: 0.0904


In [ ]:
import gc

rf_oof = np.full(len(dev_A["labels"]), np.nan)
fold_results = []

for k in range(1, 6):
    tr = dev_A["fold"] != k
    va = dev_A["fold"] == k

    # Fit preprocessing using this fold's training patients only
    missing_pct = dev_A["missing"].loc[tr, s.ALL_FEATURES].mean()
    keep_k = [c for c in s.ALL_FEATURES if missing_pct[c] < 0.99]

    fill_k = (
        dev_A["filled"].loc[tr, keep_k]
        .drop(columns=s.UNIT_COLS, errors="ignore")
        .median().to_dict()
    )
    fill_k.update({c: -1 for c in s.UNIT_COLS if c in keep_k})

    clinical_k = [c for c in s.CLINICAL if c in keep_k]
    flag_k = [c + "_measured" for c in clinical_k]
    since_k = [c + "_hours_since" for c in clinical_k]

    feature_parts = [("filled", keep_k),
                     ("flags", flag_k),
                     ("since", since_k)]

    Xtr = pd.concat([
        dev_A[key].loc[tr, cols] for key, cols in feature_parts
    ], axis=1).fillna(fill_k).astype("float32")

    Xva = pd.concat([
        dev_A[key].loc[va, cols] for key, cols in feature_parts
    ], axis=1).fillna(fill_k).astype("float32")

    rf = RandomForestClassifier(
        n_estimators=60,
        max_depth=14,
        min_samples_leaf=20,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=s.SEED
    )

    rf.fit(Xtr, dev_A["labels"][tr])
    prob = rf.predict_proba(Xva)[:, 1]
    rf_oof[va] = prob

    auc = roc_auc_score(dev_A["labels"][va], prob)
    auprc = average_precision_score(dev_A["labels"][va], prob)

    fold_results.append([k, auc, auprc])
    print(f"Fold {k}: AUROC={auc:.4f}, AUPRC={auprc:.4f}")

    del rf, Xtr, Xva
    gc.collect()

assert not np.isnan(rf_oof).any()

print("\nFive-fold results:")
display(pd.DataFrame(
    fold_results, columns=["Fold", "AUROC", "AUPRC"]
).round(4))

Fold 1: AUROC=0.8080, AUPRC=0.0949
Fold 2: AUROC=0.7888, AUPRC=0.0887
Fold 3: AUROC=0.7854, AUPRC=0.0763
Fold 4: AUROC=0.8019, AUPRC=0.0935
Fold 5: AUROC=0.7939, AUPRC=0.0934

Five-fold results:


,Fold,AUROC,AUPRC
0,1,0.8080,0.0949
1,2,0.7888,0.0887
2,3,0.7854,0.0763
3,4,0.8019,0.0935
4,5,0.7939,0.0934


In [ ]:
# Combine all five folds' validation predictions
rf_oof_df = pd.DataFrame({
    "PatientID": dev_A["patients"],
    "Hour": dev_A["hours"],
    "y_true": dev_A["labels"],
    "y_prob": rf_oof,
    "Fold": dev_A["fold"]
})

# Select the threshold using development data only
rf_cutoff_results = s.best_cutoff_table({
    "Random Forest (96 features)": rf_oof_df
})

display(rf_cutoff_results.round(4))

,Best cutoff,F1,Precision,Recall,FPR,Clinical utility,"Utility, fold mean","Utility, fold std"
Features,,,,,,,,
Random Forest (96 features),0.35,0.1204,0.0669,0.5979,0.1851,0.3627,0.3627,0.0165


In [ ]:
# Final preprocessing: fitted on all Hospital A development patients
missing_final = dev_A["missing"][s.ALL_FEATURES].mean()
keep_final = [c for c in s.ALL_FEATURES if missing_final[c] < 0.99]

fill_final = (
    dev_A["filled"][keep_final]
    .drop(columns=s.UNIT_COLS, errors="ignore")
    .median()
    .to_dict()
)
fill_final.update({c: -1 for c in s.UNIT_COLS if c in keep_final})

clinical_final = [c for c in s.CLINICAL if c in keep_final]
flags_final = [c + "_measured" for c in clinical_final]
since_final = [c + "_hours_since" for c in clinical_final]

def prepare_rf_inputs(data):
    X = pd.concat([
        data["filled"][keep_final].fillna(fill_final),
        data["flags"][flags_final],
        data["since"][since_final]
    ], axis=1).astype("float32")

    assert not X.isna().any().any()
    return X

# Same feature columns for all datasets
X_rf_final = prepare_rf_inputs(dev_A)
X_rf_test_A = prepare_rf_inputs(test_A)
X_rf_test_B = prepare_rf_inputs(test_B)

print("Features:", X_rf_final.shape[1])

# Final model: settings selected during development
rf_final = RandomForestClassifier(
    n_estimators=60,
    max_depth=14,
    min_samples_leaf=20,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=s.SEED
)

start = time.time()

rf_final.fit(X_rf_final, dev_A["labels"])

print("Final model trained successfully!")
print("Training time:", round(time.time() - start, 1), "seconds")

Features: 96
Final model trained successfully!
Training time: 101.9 seconds


In [ ]:
# Final predictions — no more training
prob_rf_A = rf_final.predict_proba(X_rf_test_A)[:, 1]
prob_rf_B = rf_final.predict_proba(X_rf_test_B)[:, 1]

# Same evaluation metrics as Logistic Regression
def evaluate_test(test, probabilities, threshold=0.35):
    y = test["labels"]
    predictions = (probabilities >= threshold).astype(int)

    utility = s.make_utility_scorer(
        test["patients"], y, test["hours"]
    )

    return {
        "AUROC": roc_auc_score(y, probabilities),
        "AUPRC": average_precision_score(y, probabilities),
        **s.threshold_metrics(y, predictions),
        "Clinical utility": utility(predictions)
    }

# Evaluate both hospitals
rf_results = pd.DataFrame({
    "Hospital A (20%)": evaluate_test(test_A, prob_rf_A),
    "Hospital B": evaluate_test(test_B, prob_rf_B)
}).T

print(rf_results.round(4).to_string())

                   AUROC   AUPRC      F1  Precision  Recall     FPR  Clinical utility
Hospital A (20%)  0.8044  0.0900  0.1177     0.0649  0.6349  0.2015            0.3783
Hospital B        0.7643  0.0524  0.1021     0.0588  0.3853  0.0885            0.2149


In [ ]:
# Save final Random Forest results
rf_summary = rf_results.reset_index().rename(
    columns={"index": "Hospital"}
)

rf_summary.insert(0, "Model", "Random Forest")
rf_summary.insert(1, "Features", 96)
rf_summary.insert(2, "Threshold", 0.35)

display(rf_summary.round(4))

rf_summary.to_csv("09_RF_Test_Results.csv", index=False)

from google.colab import files
files.download("09_RF_Test_Results.csv")

,Model,Features,Threshold,Hospital,AUROC,AUPRC,F1,Precision,Recall,FPR,Clinical utility
0,Random Forest,96,0.35,Hospital A (20%),0.8044,0.0900,0.1177,0.0649,0.6349,0.2015,0.3783
1,Random Forest,96,0.35,Hospital B,0.7643,0.0524,0.1021,0.0588,0.3853,0.0885,0.2149


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>